In [1]:
import pandas as pd
from ml_utils.src import *
from ml_utils.config import *

import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

# 1.0 - Classificação do Desempenho Baseado em Fatores Socioeconômicos Usando Random Forest

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_ESTADO_CIVIL', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
            'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df_train = pd.read_parquet(DATA_DIR, columns = colunas)
df_test = pd.read_parquet("2023", columns = colunas)

## 1.1 - Pré-Processamento dos Dados 

In [3]:
df_train = preparar_dados_forests(df_train, objetivo = 'Desempenho', n_samples = 400_000)
df_test = preparar_dados_forests(df_test, objetivo = 'Desempenho', n_samples = 100_000)

## 1.2 - Construção da Matriz X e Vetor y

In [4]:
X_train = df_train.drop(columns=['MEDIA', 'FALTOU'])
y_train = df_train['MEDIA']

X_test = df_test.drop(columns=['MEDIA', 'FALTOU'])
y_test = df_test['MEDIA']

## 1.3 - Separação em Dados de Treino e Teste

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [6]:
quantil = y_train.quantile(0.5)
y_train = (y_train >= quantil).astype(int)
y_val   = (y_val   >= quantil).astype(int)
y_test  = (y_test  >= quantil).astype(int)

## 1.4 - Treinando o modelo

In [7]:
clf = RandomForestClassifier()

clf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, clf.predict(X_train))))
print('Eval: %0.4f' % (1 - accuracy_score(y_val,  clf.predict(X_val))))

print(classification_report(y_val, clf.predict(X_val)))

Ein:  0.0362
Eval: 0.3247
              precision    recall  f1-score   support

           0       0.67      0.68      0.68     24411
           1       0.68      0.67      0.67     24463

    accuracy                           0.68     48874
   macro avg       0.68      0.68      0.68     48874
weighted avg       0.68      0.68      0.68     48874



## 1.5 Treinando com os Melhores Parâmetros

In [8]:
cv_rf = buscar_hiperparametros_rf(X_train, y_train, n_iter=30, cv=5, scoring='f1_weighted', random_state=42)

print(cv_rf.best_estimator_)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_val,  cv_rf.predict(X_val))))

print(classification_report(y_val, cv_rf.predict(X_val)))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
RandomForestClassifier(class_weight='balanced', max_depth=20,
                       max_features='log2', max_samples=1.0,
                       min_samples_leaf=20, min_samples_split=50,
                       random_state=42)
Ein:  0.2774
Eout: 0.2935
              precision    recall  f1-score   support

           0       0.70      0.73      0.71     24411
           1       0.72      0.68      0.70     24463

    accuracy                           0.71     48874
   macro avg       0.71      0.71      0.71     48874
weighted avg       0.71      0.71      0.71     48874



## 1.6 - Treinando com todos os dados de treino

In [9]:
rf = RandomForestClassifier(**cv_rf.best_params_, random_state=42)

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  rf.predict(X_test))))

print(classification_report(y_test, rf.predict(X_test)))

Ein:  0.2775
Eout: 0.3044
              precision    recall  f1-score   support

           0       0.65      0.68      0.66     29965
           1       0.74      0.71      0.72     38114

    accuracy                           0.70     68079
   macro avg       0.69      0.69      0.69     68079
weighted avg       0.70      0.70      0.70     68079



In [9]:
rf = RandomForestClassifier(max_depth=20,
                       max_features='log2', max_samples=1.0,
                       min_samples_leaf=20, min_samples_split=50, random_state=42)

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  rf.predict(X_test))))

print(classification_report(y_test, rf.predict(X_test)))

joblib.dump(rf, 'rf_desempenho.joblib')

Ein:  0.2822
Eout: 0.3143
              precision    recall  f1-score   support

           0       0.62      0.76      0.68     29965
           1       0.77      0.63      0.69     38114

    accuracy                           0.69     68079
   macro avg       0.69      0.69      0.69     68079
weighted avg       0.70      0.69      0.69     68079



['rf_desempenho.joblib']